In [9]:
import ast
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
import albumentations as A
from torch.optim import AdamW
from torch.utils.data import DataLoader
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import StratifiedKFold, GroupKFold
from torch.optim.lr_scheduler import CosineAnnealingLR
from optional_train import ResnetClassifier, DfToDataset, train_epoch, eval_model

In [10]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используется: {device}")

Используется: cuda


In [11]:
IMG_DIR = 'data/train_images/'
train_df = pd.read_csv('data/train.csv')

train_df['bbox'] = train_df['bbox'].apply(ast.literal_eval)

# 2. Распаковываем список в 4 новые числовые (float) колонки
train_df['x_min_bbox'] = train_df['bbox'].apply(lambda x: x[0])
train_df['y_min_bbox'] = train_df['bbox'].apply(lambda x: x[1])
train_df['width_bbox'] = train_df['bbox'].apply(lambda x: x[2])
train_df['height_bbox'] = train_df['bbox'].apply(lambda x: x[3])

# 3. Сразу считаем площадь (пригодится для фильтрации мусора)
train_df['area_bbox'] = train_df['width'] * train_df['height']

# 4. Считаем координаты правого нижнего угла (Pascal VOC формат для PyTorch)
train_df['x_max_bbox'] = train_df['x_min'] + train_df['width']
train_df['y_max_bbox'] = train_df['y_min'] + train_df['height']

# train_df = train_df_raw[train_df_raw['bbox'] >=]
y = train_df['bbox']

2


In [12]:
train_transforms = A.Compose([
    A.Resize(height=256, width=256),  # Здесь строго height/width
    A.RandomResizedCrop(size=(224, 224), scale=(0.8, 1.0), p=1.0),  # А здесь строго size как кортеж
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=30, p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2, p=0.5),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

# Трансформации для валидационной выборки
val_transforms = A.Compose([
    A.Resize(height=224, width=224),  # Тоже height/width
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

In [13]:
gkf = GroupKFold(n_splits=5, shuffle=True, random_state=42)

fold_metrics = []

EPOCHS = 7
for fold, (train_idx, val_idx) in enumerate(gkf.split(train_df, train_df['source'])):

    X_df_train = train_df.iloc[train_idx].reset_index(drop=True)
    X_df_val = train_df.iloc[val_idx].reset_index(drop=True)

    train_dataset = DfToDataset(X_df_train, IMG_DIR, train_transforms)
    val_dataset = DfToDataset(X_df_val, IMG_DIR, val_transforms)

    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

    model = ResnetClassifier().to(device)
    optimizer = AdamW(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss().to(device)
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

    best_fold_metric = 0
    for epoch in range(EPOCHS):
        train_loss = train_epoch(model, optimizer, train_loader, loss_fn, device)
        val_metric_dict = eval_model(model, val_loader, device)

        scheduler.step()

        print(f"Эпоха {epoch + 1}/{EPOCHS} | Loss: {train_loss:.4f}")
        print(f"Val Precision: {val_metric_dict['precision']:.4f} | Val Recall: {val_metric_dict['recall']:.4f} | Val F1: {val_metric_dict['f1']:.4f} | Val Accuracy: {val_metric_dict['accuracy']:.4f}")

        val_metric_f1 = val_metric_dict['f1']

        if val_metric_f1 > best_fold_metric:
            best_fold_metric = val_metric_f1
            torch.save(model.state_dict(), f'models/model_fold_{fold + 1}.pth')

    fold_metrics.append(best_fold_metric)
    print(f"Лучший результат фолда {fold + 1}: {best_fold_metric}\n")

print(f"Средний результат F1 меры по фолдам: {np.mean(fold_metrics)}")
